# Final Assessment — ML & Deep Learning

**Duration:** ~3 hours · **Total:** 100 points (50 math + 50 coding)

Complete all sections below. For Part B2, choose **either** the diffusion sampler track **or** the KV-cache track (not both).

Before submitting: **Kernel → Restart & Run All**.

In [ ]:
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PATH = Path("../data/assessment_synthetic.csv")
RNG = np.random.default_rng(0)
torch.manual_seed(0)

print(f"Device: {DEVICE}")

---
# Part A — Mathematics (50 points)

Write your answers in the markdown cells. Show key steps.

## A1 — Chain rule and gradients (10 pts)

Let $f(x, y) = \sin(x y) + x^2$ with $x, y \in \mathbb{R}$.

1. Compute $\partial f / \partial x$ and $\partial f / \partial y$.
2. Evaluate both at $(x, y) = (1, 2)$.
3. Suppose $x = g(t) = t^2$ and $y = h(t) = 3t$. Use the chain rule to find $df/dt$ as a function of $t$, and evaluate at $t = 1$.

**Your answer:**

*Write answer here.*

## A2 — Backpropagation sketch (10 pts)

Consider a scalar loss $\mathcal{L} = \frac{1}{2}(\hat{y} - y)^2$ where $\hat{y} = w_2 \, \sigma(w_1 x + b_1) + b_2$, $\sigma$ is the logistic sigmoid, and $w_1, w_2, b_1, b_2, x, y$ are scalars.

1. Draw the computation graph (brief sketch or bullet list of dependencies).
2. Derive $\partial \mathcal{L} / \partial w_1$ in terms of $x, y, w_1, w_2, b_1, b_2$.

**Your answer:**

*Write answer here.*

## A3 — KL divergence and ELBO (10 pts)

For two univariate Gaussians $p = \mathcal{N}(\mu_p, \sigma_p^2)$ and $q = \mathcal{N}(\mu_q, \sigma_q^2)$:

1. Write the closed-form $\mathrm{KL}(q \| p)$.
2. In a VAE, the ELBO is $\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - \mathrm{KL}(q_\phi(z|x) \| p(z))$. Explain in 2–3 sentences why maximizing the ELBO is equivalent to minimizing a reconstruction error plus a regularizer on the latent distribution.

**Your answer:**

*Write answer here.*

## A4 — Forward diffusion and score matching (15 pts)

A DDPM forward process is $q(x_t | x_0) = \mathcal{N}(\sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t) I)$ with $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$.

1. Show that $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon$ with $\epsilon \sim \mathcal{N}(0, I)$.
2. The training objective predicts noise: $\mathbb{E}_{t, x_0, \epsilon}\|\epsilon_\theta(x_t, t) - \epsilon\|^2$. Relate the optimal $\epsilon_\theta$ to the score $\nabla_{x_t} \log q(x_t)$ (one or two sentences).
3. Why does the signal-to-noise ratio decrease as $t$ increases?

**Your answer:**

*Write answer here.*

## A5 — Scaled dot-product attention (15 pts)

Given queries $Q \in \mathbb{R}^{n \times d_k}$, keys $K \in \mathbb{R}^{m \times d_k}$, values $V \in \mathbb{R}^{m \times d_v}$:

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

1. What are the shapes of $Q K^\top$ and the output?
2. Why divide by $\sqrt{d_k}$? (Hint: variance of dot products when entries are i.i.d. with unit variance.)
3. In causal (decoder-only) language modeling, how is the softmax input masked before applying softmax?

**Your answer:**

*Write answer here.*

---
# Part B — Coding (50 points)

## B1 — MLP classifier (25 pts)

Train a multi-layer perceptron on `../data/assessment_synthetic.csv` (binary classification).

**Requirements:** train/val split, ≥2 hidden layers, report validation accuracy, plot training loss.

In [ ]:
def load_assessment_data(path: Path = DATA_PATH):
    data = np.loadtxt(path, delimiter=",", skiprows=1)
    X, y = data[:, :-1].astype(np.float32), data[:, -1].astype(np.int64)
    return torch.from_numpy(X), torch.from_numpy(y)


class MLPClassifier(nn.Module):
    def __init__(self, in_dim: int, hidden: int = 64, n_layers: int = 2, n_classes: int = 2):
        super().__init__()
        layers = []
        dim = in_dim
        for _ in range(n_layers):
            layers += [nn.Linear(dim, hidden), nn.ReLU()]
            dim = hidden
        layers.append(nn.Linear(dim, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_mlp(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 30,
    lr: float = 1e-3,
):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(xb)
        history["train_loss"].append(total_loss / len(train_loader.dataset))

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb).argmax(dim=1)
                correct += (pred == yb).sum().item()
                total += len(yb)
        history["val_acc"].append(correct / total)

    return history


# --- Your training run below ---
X, y = load_assessment_data()
dataset = TensorDataset(X, y)
n_val = int(0.2 * len(dataset))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(0))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)

mlp = MLPClassifier(in_dim=X.shape[1], hidden=64, n_layers=2)
history = train_mlp(mlp, train_loader, val_loader, epochs=30)

print(f"Final validation accuracy: {history['val_acc'][-1]:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(history["train_loss"])
ax[0].set(xlabel="epoch", ylabel="train loss", title="Training loss")
ax[1].plot(history["val_acc"])
ax[1].set(xlabel="epoch", ylabel="val accuracy", title="Validation accuracy")
plt.tight_layout()
plt.show()

## B2 — Choose one track (25 pts)

Complete **either** §B2a (diffusion sampler) **or** §B2b (KV cache). Mark your choice clearly.

### B2a — Diffusion reverse sampler (DDPM-style)

Implement a 1-D reverse diffusion sampler for a **known** score of a mixture of Gaussians. Use a linear noise schedule with $T$ steps.

**Deliverables:** sample histogram, brief comment on sample quality.

In [ ]:
# B2 TRACK: diffusion (delete or skip B2b if using this track)

def true_score_1d(x: torch.Tensor) -> torch.Tensor:
    """Score of 0.5 N(-2, 0.5) + 0.5 N(2, 0.5)."""
    w1, m1, s1 = 0.5, -2.0, 0.5
    w2, m2, s2 = 0.5, 2.0, 0.5
    n1 = torch.exp(-0.5 * ((x - m1) / s1) ** 2) / (s1 * math.sqrt(2 * math.pi))
    n2 = torch.exp(-0.5 * ((x - m2) / s2) ** 2) / (s2 * math.sqrt(2 * math.pi))
    p = w1 * n1 + w2 * n2
    s1_score = n1 * (-(x - m1) / s1**2)
    s2_score = n2 * (-(x - m2) / s2**2)
    return (w1 * s1_score + w2 * s2_score) / (p + 1e-8)


def ddpm_reverse_sample(score_fn, T: int = 200, n_samples: int = 2000, sigma_max: float = 2.0):
    """Langevin-style reverse sampler using true score (toy DDPM)."""
    betas = torch.linspace(1e-4, 0.02, T)
    alphas = 1.0 - betas
    alpha_bar = torch.cumprod(alphas, dim=0)
    x = torch.randn(n_samples) * sigma_max

    for t in reversed(range(T)):
        ab = alpha_bar[t]
        score = score_fn(x)
        noise = torch.randn_like(x) if t > 0 else torch.zeros_like(x)
        # Reverse update (simplified; students may refine)
        x = (x + (1 - ab) * score) / torch.sqrt(alphas[t]) + torch.sqrt(betas[t]) * noise
    return x


samples = ddpm_reverse_sample(true_score_1d, T=200).numpy()
plt.hist(samples, bins=50, density=True, alpha=0.7, label="samples")
xs = np.linspace(-5, 5, 400)
pdf = 0.5 * np.exp(-0.5 * ((xs + 2) / 0.5) ** 2) / (0.5 * np.sqrt(2 * np.pi))
pdf += 0.5 * np.exp(-0.5 * ((xs - 2) / 0.5) ** 2) / (0.5 * np.sqrt(2 * np.pi))
plt.plot(xs, pdf, "k--", label="true mixture")
plt.legend()
plt.title("B2a: reverse diffusion samples vs true density")
plt.show()

*Brief analysis of B2a sample quality:*

### B2b — KV-cache autoregressive decoding

Implement greedy decoding for a tiny causal transformer **with** and **without** KV caching. Report tokens/sec speedup.

**Deliverables:** timing comparison table/plot, 20-token greedy continuation of a fixed prompt.

In [ ]:
# B2 TRACK: KV cache (delete or skip B2a if using this track)

class TinyCausalTransformer(nn.Module):
    def __init__(self, vocab: int = 32, d_model: int = 64, n_heads: int = 4, n_layers: int = 2):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model, n_heads, dim_feedforward=128, batch_first=True
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, vocab)
        self.d_model = d_model

    def forward(self, idx: torch.Tensor, past_kv=None):
        # past_kv unused in this minimal stub; students implement caching
        x = self.embed(idx)
        T = x.size(1)
        mask = torch.triu(torch.ones(T, T, device=idx.device) * float("-inf"), diagonal=1)
        h = self.blocks(x, mask=mask)
        return self.head(h)


def greedy_decode(model, prompt: torch.Tensor, max_new: int = 20, use_cache: bool = False):
    model.eval()
    seq = prompt.clone()
    cache = None
    with torch.no_grad():
        for _ in range(max_new):
            if use_cache and cache is not None:
                logits = model(seq[:, -1:], past_kv=cache)
            else:
                logits = model(seq)
            next_id = logits[:, -1].argmax(dim=-1, keepdim=True)
            seq = torch.cat([seq, next_id], dim=1)
            # TODO: update cache when use_cache=True
    return seq


def benchmark_decode(model, prompt, max_new=20, repeats=10):
    times = {}
    for use_cache in (False, True):
        t0 = time.perf_counter()
        for _ in range(repeats):
            greedy_decode(model, prompt, max_new=max_new, use_cache=use_cache)
        times[use_cache] = (time.perf_counter() - t0) / repeats
    return times


model = TinyCausalTransformer().to(DEVICE)
prompt = torch.randint(0, 32, (1, 8), device=DEVICE)
times = benchmark_decode(model, prompt, max_new=20, repeats=5)
tok_per_sec = {k: 20 / v for k, v in times.items()}
print(f"No cache: {times[False]:.4f}s ({tok_per_sec[False]:.1f} tok/s)")
print(f"With cache: {times[True]:.4f}s ({tok_per_sec[True]:.1f} tok/s)")
print(f"Speedup: {times[False] / max(times[True], 1e-9):.2f}x")

out = greedy_decode(model, prompt, max_new=20, use_cache=False)
print("Greedy continuation (token ids):", out[0].tolist())

*Implement KV caching in `greedy_decode` and describe your speedup here.*

---
## Reflection

Which B2 track did you choose and why? (One paragraph.)

*Write reflection here.*